In [ ]:
import sys, os
path = os.path.abspath('/gpfs/data1/vclgp/decontot/repos/gedih3/src')
sys.path.insert(0, path)

In [ ]:
from dask.distributed import Client
client = Client(threads_per_worker=1, n_workers=20, memory_limit='20GB')
print(client.dashboard_link)

In [ ]:
import gedih3.gh3driver as gh3
from gedih3.cliutils import parse_region

region = parse_region('/gpfs/data1/vclgp/decontot/data/vector/other_boundaries/md.shp')
query_str = "quality_flag_l2a == 1 & l4_quality_flag_l4a == 1 & l2_quality_flag_l4a == 1 & wsci_quality_flag_l4c == 1"

ddf = gh3.gh3_load(
    columns=['wsci_l4c', 'pai_z_000_l2b', 'agbd_l4a', 'geometry', 'rh_098_l2a', 'datetime', 'h3_03'],
    region=region,
    query=query_str,
    gh3_dir="/gpfs/data1/vclgp/data/iss_gedi/h3_mock/database"
)

ddf

In [ ]:
ddf['h3_03'] = ddf['h3_03'].astype(str)
ddf

In [ ]:
import geopandas as gpd
gpd.read_parquet('/gpfs/data1/vclgp/decontot/repos/gedih3/tmp/tmp/maryland/')


In [ ]:
write_task = ddf.to_parquet('/gpfs/data1/vclgp/decontot/repos/gedih3/tmp/tmp/maryland', 
                            write_metadata_file=True,
                            write_index=True, 
                            overwrite=True, 
                            compression='zstd', 
                            partition_on=[f'h3_03'], 
                            compute=True,
                            write_empty_partitions=False
                            )

In [ ]:
ddf.h3_03.unique().compute().values

In [1]:
import geopandas as gpd
df = gpd.read_parquet('/gpfs/data1/vclgp/decontot/repos/gedih3/tmp/tmp/maryland/_common_metadata')
df.columns.values

array(['datetime', 'rh_098_l2a', 'geometry', 'wsci_l4c', 'pai_z_000_l2b',
       'agbd_l4a', 'h3_03'], dtype=object)

In [2]:
import dask_geopandas
ddf = dask_geopandas.read_parquet('/gpfs/data1/vclgp/decontot/repos/gedih3/tmp/tmp/maryland')
ddf

,datetime,rh_098_l2a,geometry,wsci_l4c,pai_z_000_l2b,agbd_l4a,h3_03
npartitions=47,,,,,,,
,datetime64[ns],float32,geometry,float32,float32,float32,category[known]
,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...
,...,...,...,...,...,...,...


In [ ]:
gh3_extract \
-o /gpfs/data1/vclgp/decontot/repos/gedih3/tmp/tmp/maryland \
-g -t -l2a rh_098 -l2b pai_z_000 -l4a agbd -l4c wsci -y \
-r /gpfs/data1/vclgp/decontot/data/vector/other_boundaries/md.shp

In [ ]:
f = '/gpfs/data1/vclgp/data/iss_gedi/h3_mock/database/_metadata'
pq.read_metadata(f)

In [ ]:
ISO3_URL = "https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/country_shapes/exports/geojson/"
gdf = gpd.read_file(ISO3_URL)
gdf

In [ ]:
import geopandas as gpd
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))


In [ ]:
import os
import dask.dataframe
import dask_geopandas
from dask.distributed import Client

tmp = '/gpfs/data1/vclgp/decontot/temp/dask-tmp'
client = Client(n_workers=10, threads_per_worker=1, memory_limit='30GB', local_directory=tmp)
client#.shutdown()  # Uncomment to stop the client

In [ ]:
from shapely.geometry import box 
region = box(-95.0, 25.0, -65.0, 50.0)

In [ ]:
import gedih3.gh3driver as gh3

In [ ]:
cols = ['shot_number', 'elev_lowestmode_l2a', 'wsci_l4c', 'agbd_l4a', 'h3_03']

In [ ]:
q = 'wsci_l4c >= 0 and agbd_l4a > 0'
h3dir = '/gpfs/data1/vclgp/data/iss_gedi/h3_mock/database/'

gh3_df = gh3.gh3_load(region=region, columns=cols, query=q, gh3_dir=h3dir)
gh3_df

In [ ]:
import pandas as pd

# agg = {'wsci_l4c': 'mean', 'elev_lowestmode_l2a': 'mean', 'agbd_l4a': ['mean', 'std']}
agg = lambda x: pd.Series((x.wsci_l4c / x.agbd_l4a).mean())
res = 5

adf = gh3.gh3_aggregate(gh3_df, target_res=res, agg=agg)
adf

In [ ]:
import glob
h3dir = '/gpfs/data1/vclgp/data/iss_gedi/h3_mock/database/'
h3dirs = glob.glob(os.path.join(h3dir, 'h3_*/'))

In [ ]:
import geopandas as gpd
from gedih3.gh3driver import intersect_h3_geometries, gh3_read_meta

def _load_map(d, columns=None):
    files = glob.glob(os.path.join(d, '**/*.parquet'), recursive=True)
    return gpd.read_parquet(files, columns=columns)

def gh3_load_from_map(columns=None, region=None, query=None, gh3_dir=h3dir):
    h3_part = gh3_read_meta("h3_partition_level", gh3_root_dir=gh3_dir)
    h3_part_col = f"h3_{h3_part:02d}"
    h3_ids = gh3_read_meta("h3_partition_ids", gh3_root_dir=gh3_dir)
    
    out_cols = None
    if columns is not None:
        if h3_part_col not in columns:
            columns.append(h3_part_col)
            
        if 'geometry' not in columns:
            columns.append('geometry')

        out_cols = columns.copy()
        
        if query is not None:
            available_cols = gh3_read_meta("h3_columns", gh3_root_dir=gh3_dir)
            q_cols = [col for col in available_cols if col in query]
            columns = list(set(columns + q_cols))        

    if region is not None:
        h3_ids = intersect_h3_geometries(region, h3_ids=h3_ids)
        h3_dirs = [os.path.join(gh3_dir, f"{h3_part_col}={i}") for i in h3_ids]
    else:
        h3_dirs = glob.glob(os.path.join(gh3_dir, f"{h3_part_col}=*/"))

    _meta = _load_map(h3_dirs[0], columns=columns)
    ddf = dask.dataframe.from_map(_load_map, h3_dirs, columns=columns, meta=_meta)
    ddf = dask_geopandas.from_dask_dataframe(ddf, geometry='geometry')

    if query is not None:
        ddf = ddf.query(query)
        if out_cols is not None:
            ddf = ddf[out_cols]

    return ddf

ddf = gh3_load_from_map(region=region, columns=cols, query=q, gh3_dir=h3dir)
ddf

In [ ]:
from gedih3.gh3driver import gh3_aggregate_func, gh3_part_from_df, gh3_add_geometry
from gedih3.h3utils import fix_h3_geometry

def gh3_aggregate_from_map(gh3_df, target_res=5, agg='mean', columns=None, query=None, add_geometry=True):
    _meta = gh3_aggregate_func(df=gh3_df._meta, res=target_res, agg=agg, cols=columns)

    if query is not None:
        gh3_df = gh3_df.query(query)

    agg_df = gh3_df.map_partitions(gh3_aggregate_func, res=target_res, agg=agg, cols=columns, meta=_meta)
    agg_df = agg_df.set_index(f"h3_{target_res:02d}", sort=False)
    
    if add_geometry:
        _gmeta = gpd.GeoDataFrame(columns=agg_df._meta.columns.tolist() + ['geometry'], geometry='geometry', crs=4326)
        agg_df = agg_df.map_partitions(gh3_add_geometry, meta=_gmeta)

    return agg_df

adf = gh3_aggregate_from_map(ddf, target_res=5, agg=agg, add_geometry=True)
adf

In [ ]:
import gedih3 as gh3
tmp = '//gpfs/data1/vclgp/data/iss_gedi/h3_mock/tmp/duckdb'
ddb = gh3.sqlutils.init_duckdb(temp_directory=tmp)

In [ ]:
def data_spec(hex_id=None, year=None):
    db_path = '/gpfs/data1/vclgp/data/iss_gedi/h3_mock/database'
    h3_part = "*"
    year_part = "*"
    if hex_id is not None:
        h3_part = f"{hex_id}"
    if year is not None:
        year_part = f"year={year}"
    return f"{db_path}/{h3_part}/{year_part}/*.parquet"

In [ ]:
ddb.sql(f"""
    SELECT *
    FROM read_parquet('{data_spec()}')
    WHERE  {q}
    LIMIT 10
""")